# Positioning table (3-seed, all methods) + generality tests (Theil, log-barrier)

Closes review items #1, #2, #3, #5:

- **#3**: replaces single-seed Gated/L1 rows in the positioning table with the same 3-seed protocol Top-k and Rate-KL already get.
- **#5**: adds `mean_active_magnitude` for Rate-KL (and every method), closing the missing-control gap.
- **#1**: Theil index through the same purity/importance-concentration diagnostic used on Gini -- second instance of the general hub-pathology claim.
- **#2**: log-barrier + budget penalty (identity-tracked, boundary-escalating, but NOT KL-shaped) -- second instantiation of the design principle behind Rate-KL, testing whether the principle is separable from the specific formula.

A gradient-magnitude sanity check (Theil at a hub point vs.\ a genuinely varied selective point) already confirmed the vanishing-gradient prediction analytically before this notebook runs any training -- see `experiments/theil_diagnosis.py`'s docstring.

**Before running:** Runtime -> Change runtime type -> GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (go to Runtime > Change runtime type > GPU)')

In [ ]:
!git clone https://github.com/willkn/SAE-Gini.git
%cd SAE-Gini/experiments

## Part 1: Full positioning replication (3 seeds, Fashion-MNIST)
Trains Tuned L1, Gated, Gini, Top-k, Rate-KL (lambda=0.001, 0.01) per seed -- 5 models x 3 seeds = 15 trainings.

In [ ]:
!python run_full_positioning_replication.py --dataset fashion_mnist --seeds 0 1 2 --rho 0.09 --lambdas 0.001 0.01

In [ ]:
import json
with open('results/full_positioning/fashion_mnist_summary.json') as f:
    summary = json.load(f)
print(f"{'Model':16s} {'Sparsity':>11s} {'MSE':>13s} {'Magnitude':>14s} {'MeanPurity':>14s} {'Top10Purity':>14s} {'Ablate':>14s} {'Clamp':>14s}")
for model, m in summary.items():
    def fmt(key, prec=4):
        v = m.get(key)
        return f"{v['mean']:.{prec}f}+/-{v['std']:.{prec}f}" if v else 'n/a'
    print(f"{model:16s} {fmt('relative_sparsity', 3):>11s} {fmt('mse'):>13s} {fmt('mean_active_magnitude', 3):>14s} "
          f"{fmt('mean_purity', 3):>14s} {fmt('mean_purity_top10pct_by_importance', 3):>14s} "
          f"{fmt('steering_impact_ablate'):>14s} {fmt('steering_impact_clamp'):>14s}")

## Part 2: Theil index generality test (Fashion-MNIST)
Second instance of the hub-pathology claim: does a differently-shaped inequality measure (Theil, not Gini) produce the same hub signature?

In [ ]:
!python theil_diagnosis.py --dataset fashion_mnist --seed 0 --lambdas 0.01 0.05 0.1

## Part 3: Log-barrier second instantiation (Fashion-MNIST)
Tests whether the identity-tracked + boundary-escalating DESIGN PRINCIPLE (not the specific KL formula) is what avoids the hub pathology, using a functionally unrelated penalty (interior-point log-barrier) that shares only those two properties.

In [ ]:
!python rate_barrier_sae.py --dataset fashion_mnist --seed 0 --rho 0.09 --lambda-pairs 0.001,0.1 0.005,0.5 0.01,1.0

In [ ]:
!zip -r positioning_and_generality_results.zip results/full_positioning results/theil_diagnosis results/rate_barrier
from google.colab import files
files.download('positioning_and_generality_results.zip')